In [ ]:
# !pip install langchain langchain-community pypdf langchain-core
# !pip install langchain-chroma chromadb
# !pip install langchain-openai
# !pip install langchain-huggingface
# !pip install pymupdf
# !pip install transformers accelerate openai
# !pip install pymupdf4llm

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
os.environ['PYTHONUTF8'] = '1'

import re
import unicodedata
import pymupdf4llm
from glob import glob

In [ ]:
def fix_text_quality(text: str) -> str:
    text = unicodedata.normalize('NFKC', text)

    # 직접 보정
    broken_words = {
        r'\bO icial\b': 'Official',
        r'\bO icials\b': 'Officials',
        r'\bO icially\b': 'Officially',
        r'\bspeci ic\b': 'specific',
        r'\bSpeci ic\b': 'Specific',
        r'\bspeci ied\b': 'specified',
        r'\bSpeci ied\b': 'Specified',
        r'\bspeci ication\b': 'specification',
        r'\bcoef icient\b': 'coefficient',
        r'\bef ective\b': 'effective',
        r'\bEf ective\b': 'Effective',
        r'\bef iciently\b': 'efficiently',
        r'\bsuf icient\b': 'sufficient',
        r'\bdif erent\b': 'different',
        r'\bdif iculty\b': 'difficulty',
        r'\baf ect\b': 'affect',
        r'\baf iliate\b': 'affiliate',
    }
    for broken, fixed in broken_words.items():
        text = re.sub(broken, fixed, text)

    return text

In [ ]:
def remove_noise(text: str) -> str:
    """머리글/꼬리말/목차/메타정보 제거"""

    # 줄바꿈 정규화 (\r\n → \n) 
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    text = re.sub(
        r'\n2026 Formula 1:[^\n]+©\d{4} Fédération Internationale de l\'Automobile[^\n]*\n',
        '\n', text
    )

    # 날짜 줄 제거
    text = re.sub(r'\n\d{1,2} \w+ \d{4}[^\n]*\n', '\n', text)

    # SECTION X: ...  제거
    text = re.sub(r'\nSECTION [A-Z]: [A-Z &]+\n', '\n', text)

    # B 0 형태 제거
    text = re.sub(r'\n[A-Z0-9] [A-Z0-9]\s*\n', '\n', text)

    # A1 형태 제거
    text = re.sub(r'\n[A-Z]\d+\s*\n', '\n', text)

    # Issue 번호 제거
    text = re.sub(r'\nIssue \d+\s*\n', '\n', text)

    # FIA 로고 그림 제거
    text = re.sub(
        r'\n\*\*==> picture \[\d+ x \d+\] intentionally omitted <==\*\*\n'
        r'(\n)?'
        r'\*\*----- Start of picture text -----\*\*<br>\n'
        r'.+?<br>\*\*----- End of picture text -----\*\*<br>\n',
        '\n', text, flags=re.DOTALL
    )

    # 테이블 형태 섹션 헤더 제거
    text = re.sub(
        r'\n\|SECTION [A-Z]: [^\|]+\|[^\|]+\|\n\|---\|---\|\n',
        '\n', text
    )

    # 문서 최상단 SECTION X: ... 제거
    text = re.sub(r'^SECTION [A-Z]: [A-Z ]+\n', '', text)

    # Version/Issue/Status/Date 제거
    text = re.sub(r'\nVersion: Issue \d+[^\n]+\n', '\n', text)

    # CONVENTION 제거
    text = re.sub(
        r'## \*\*CONVENTION:\*\*.*?(?=## \*\*)',
        '', text, flags=re.DOTALL
    )

    # CONTENTS 제거
    text = re.sub(
        r'## \*\*CONTENTS:\*\*.*?(?=## \*\*(?:PREAMBLE|ARTICLE\s+[A-Z]))',
        '', text, flags=re.DOTALL
    )

    # Advisory Committee, Governance 제거
    text = re.sub(r'_Advisory Committee:.*?_\n', '\n', text)
    text = re.sub(r'_Governance:.*?_\n', '\n', text)

    # 연속 빈 줄 정리
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

In [ ]:
files = glob("./data/fia_2026_f1_regulations_*.pdf")
print(f"발견된 PDF: {len(files)}개\n")

md_files = []

for pdf in files:
    print(f"변환 중: {pdf}")

    md = pymupdf4llm.to_markdown(pdf)

    # 취소선 제거
    before_strikethrough = len(re.findall(r'~~.+?~~', md))
    md = re.sub(r'~~.+?~~', '', md)

    # 텍스트 품질 보정
    md = fix_text_quality(md)

    # 노이즈 제거
    md = remove_noise(md)

    md_file = pdf.replace('.pdf', '.md')
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write(md)

    print(f"{md_file}")
    print(f"취소선 제거: {before_strikethrough}개\n")
    md_files.append(md_file)

print("=" * 60)
print(f"{len(md_files)}개 파일 변환 완료!")
print("=" * 60)